# Sales & Demand Forecasting — Monthly

End-to-end walkthrough: data cleaning -> feature engineering -> modeling -> evaluation -> visualization.

Make sure `data/superstore.csv` exists before running (see README.md).

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src import data_cleaning, feature_engineering, modeling, evaluation, visualization
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Data cleaning & aggregation to monthly

In [3]:
monthly = data_cleaning.run()
monthly.tail()

[clean] Loaded 9994 raw rows
[clean] 9994 rows after cleaning
[clean] Aggregated to 48 monthly data points (2014-01-01 to 2017-12-01)
[clean] Saved -> D:\toy\sales_forecasting\outputs\monthly_sales.csv


,date,sales,is_outlier
43,2017-08-01,"63,120.89",False
44,2017-09-01,"87,866.65",False
45,2017-10-01,"77,776.92",False
46,2017-11-01,"118,447.82",False
47,2017-12-01,"83,829.32",False


## 2. Time-based feature engineering

Calendar features, lag features (1/2/3/12 months), rolling means/stds.

In [ ]:
features = feature_engineering.run()
features.tail()

## 3. Modeling

Three models compared: naive baseline, SARIMA, and XGBoost/RandomForest regression.
Train/test split is time-ordered (last 6 months held out) — no shuffling.

In [5]:
predictions = modeling.run()
predictions

[model] Fit baseline, SARIMA, and xgboost regression
[model] Saved predictions -> D:\toy\sales_forecasting\outputs\predictions.csv
      date     actual  baseline_naive    sarima  regression_xgboost
2017-07-01  45,264.42       39,261.96 47,968.74           32,746.56
2017-08-01  63,120.89       31,115.37 45,452.21           33,866.82
2017-09-01  87,866.65       73,410.02 82,800.72           77,081.93
2017-10-01  77,776.92       59,687.75 60,085.06           64,529.02
2017-11-01 118,447.82       79,411.97 89,772.10           79,179.88
2017-12-01  83,829.32       96,999.04 98,693.41           94,984.70


,date,actual,baseline_naive,sarima,regression_xgboost
0,2017-07-01,"45,264.42","39,261.96","47,968.74","32,746.56"
1,2017-08-01,"63,120.89","31,115.37","45,452.21","33,866.82"
2,2017-09-01,"87,866.65","73,410.02","82,800.72","77,081.93"
3,2017-10-01,"77,776.92","59,687.75","60,085.06","64,529.02"
4,2017-11-01,"118,447.82","79,411.97","89,772.10","79,179.88"
5,2017-12-01,"83,829.32","96,999.04","98,693.41","94,984.70"


## 4. Evaluation & error analysis

MAE, RMSE, MAPE for each model, plus residuals for the best one.

In [6]:
metrics, residuals = evaluation.run()
metrics

[eval] Model comparison (sorted by MAPE, lower = better):
             model       MAE      RMSE  MAPE_%
            sarima 14,445.10 16,839.44   17.40
regression_xgboost 19,371.31 22,251.32   24.96
    baseline_naive 20,459.89 23,430.16   25.39

[eval] Best model: sarima
[eval] Saved -> D:\toy\sales_forecasting\outputs\metrics.csv, D:\toy\sales_forecasting\outputs\residuals.csv


,model,MAE,RMSE,MAPE_%
0,sarima,"14,445.10","16,839.44",17.40
1,regression_xgboost,"19,371.31","22,251.32",24.96
2,baseline_naive,"20,459.89","23,430.16",25.39


In [7]:
residuals

,date,actual,sarima,residual,abs_pct_error,best_model
0,2017-07-01,"45,264.42","47,968.74","-2,704.32",5.97,sarima
1,2017-08-01,"63,120.89","45,452.21","17,668.68",27.99,sarima
2,2017-09-01,"87,866.65","82,800.72","5,065.93",5.77,sarima
3,2017-10-01,"77,776.92","60,085.06","17,691.86",22.75,sarima
4,2017-11-01,"118,447.82","89,772.10","28,675.72",24.21,sarima
5,2017-12-01,"83,829.32","98,693.41","-14,864.10",17.73,sarima


## 5. Business-friendly visual output

In [8]:
summary = visualization.run()
summary

[viz] Saved chart -> D:\toy\sales_forecasting\outputs\forecast_chart.png
[viz] Saved chart -> D:\toy\sales_forecasting\outputs\forecast_full_history.png
[viz] Saved summary table -> D:\toy\sales_forecasting\outputs\summary_table.csv
      date     actual  forecast  expected_range_low  expected_range_high model_used
2017-07-01  45,264.00 47,969.00           39,622.00            56,315.00     sarima
2017-08-01  63,121.00 45,452.00           37,544.00            53,361.00     sarima
2017-09-01  87,867.00 82,801.00           68,393.00            97,208.00     sarima
2017-10-01  77,777.00 60,085.00           49,630.00            70,540.00     sarima
2017-11-01 118,448.00 89,772.00           74,152.00           105,392.00     sarima
2017-12-01  83,829.00 98,693.00           81,521.00           115,866.00     sarima

[viz] Headline: best model is 'sarima' (MAPE 17.4%) — on average forecasts are within 17.4% of actual sales.


,date,actual,forecast,expected_range_low,expected_range_high,model_used
0,2017-07-01,"45,264.00","47,969.00","39,622.00","56,315.00",sarima
1,2017-08-01,"63,121.00","45,452.00","37,544.00","53,361.00",sarima
2,2017-09-01,"87,867.00","82,801.00","68,393.00","97,208.00",sarima
3,2017-10-01,"77,777.00","60,085.00","49,630.00","70,540.00",sarima
4,2017-11-01,"118,448.00","89,772.00","74,152.00","105,392.00",sarima
5,2017-12-01,"83,829.00","98,693.00","81,521.00","115,866.00",sarima


## 6. Notes / next steps

- If MAPE is high, check: enough history (24+ months ideal), outlier months, or try tuning SARIMA (p,d,q) via AIC grid search.
- Try filtering to a single `Category`/`Region` in `src/data_cleaning.py` if the aggregate series is too noisy.
- Export `outputs/summary_table.csv` and `outputs/forecast_chart.png` directly into your report or Power BI.